[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/solutions/checkpoints/08_knn.ipynb)

# ✅ Checkpoint — 08 · K-Nearest Neighbors

> 📗 **BẢN LỜI GIẢI** — chỉ mở sau khi đã tự làm checkpoint.

**Cách làm:**
1. Trả lời quiz bằng cách điền vào dict đáp án rồi chạy cell chấm điểm. *Không mở `solutions/` trước khi nộp.*
2. Hoàn thành các task code trong vùng `### START CODE HERE ###` … `### END CODE HERE ###`, chạy cell test.
3. Phần phân tích: viết câu trả lời, rồi **tự chấm trung thực** theo rubric.
4. Chạy cell cuối để xem **STATUS: ✅ PASS / ❌ REVIEW** — kết quả được lưu vào `progress/progress.json`.

**Điều kiện PASS:** Theory Quiz ≥ 80%, Math Quiz ≥ 80%, mọi task code PASS, rubric phân tích ≥ 80%.


In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/solutions/checkpoints

In [ ]:
from ml_course.checkpoint import Checkpoint
from ml_course.checks import check_close, check_equal, check_greater, check_less, check_shape, check_true
import numpy as np

cp = Checkpoint("08_knn", record=False)

## Phần 1 — Theory Quiz

1. **[t01]** Vì sao KNN được gọi là 'lazy learner'?
   - **(a)** Vì nó học chậm
   - **(b)** Vì 'train' chỉ là ghi nhớ dữ liệu; mọi tính toán dồn vào lúc dự đoán
   - **(c)** Vì nó chỉ dùng một phần dữ liệu
   - **(d)** Vì nó không cần chọn hyperparameter

2. **[t02]** KNN là model non-parametric: 'kích thước' model tăng theo số mẫu train. *(Đúng/Sai)*

3. **[t03]** Tăng k trong KNN (giữ dữ liệu cố định) thường làm:
   - **(a)** Bias giảm, variance tăng
   - **(b)** Bias tăng, variance giảm
   - **(c)** Cả hai cùng giảm
   - **(d)** Không ảnh hưởng

4. **[t04]** Với k = 1 (không có điểm trùng nhau mà khác nhãn), accuracy trên tập train là:
   - **(a)** Khoảng 50%
   - **(b)** 100%
   - **(c)** Bằng accuracy test
   - **(d)** Không xác định

5. **[t05]** Khi nào nên dùng cosine distance thay cho Euclid?
   - **(a)** Khi quan tâm hướng của vector, bỏ qua độ dài (văn bản TF-IDF, embedding)
   - **(b)** Khi dữ liệu có ít chiều
   - **(c)** Khi feature là số nguyên
   - **(d)** Không bao giờ

6. **[t06]** Độ phức tạp dự đoán MỘT điểm của KNN brute force với n điểm train, d chiều:
   - **(a)** O(1)
   - **(b)** O(d)
   - **(c)** O(n·d)
   - **(d)** O(n²)

7. **[t07]** KD-tree luôn nhanh hơn brute force, bất kể số chiều. *(Đúng/Sai)*

8. **[t08]** KNN có trọng số khoảng cách (weights='distance') làm gì?
   - **(a)** Láng giềng càng gần càng có tiếng nói lớn (w = 1/d)
   - **(b)** Bỏ qua láng giềng xa nhất
   - **(c)** Tự động chọn k
   - **(d)** Scale feature theo khoảng cách

9. **[t09]** Vì sao KNN gần như bắt buộc phải scaling?
   - **(a)** Để train nhanh hơn
   - **(b)** Vì khoảng cách cộng gộp mọi feature — feature có thang đo lớn sẽ áp đảo
   - **(c)** Vì sklearn yêu cầu
   - **(d)** Để tránh overfit

10. **[t10]** Model nào cho kết quả THAY ĐỔI khi đổi đơn vị một feature từ 'triệu' sang 'nghìn' (không scale)? *(chọn tất cả đáp án đúng)*
   - **(a)** KNN
   - **(b)** Decision Tree
   - **(c)** Linear Regression không regularization (dự đoán)
   - **(d)** KNN với cosine distance trên dữ liệu nhiều chiều

11. **[t11]** Cách đúng để chọn k là:
   - **(a)** Chọn k cho accuracy train cao nhất
   - **(b)** Chọn k bằng cross-validation trên tập train (scaler nằm trong Pipeline)
   - **(c)** Chọn k cho accuracy test cao nhất
   - **(d)** Luôn chọn k = 5

12. **[t12]** Với bài toán 2 lớp, chọn k lẻ giúp tránh hoà phiếu (với weights='uniform'). *(Đúng/Sai)*

13. **[t13]** Trong không gian rất nhiều chiều, điều gì xảy ra với khoảng cách giữa các điểm ngẫu nhiên?
   - **(a)** Khoảng cách gần nhất và xa nhất gần như bằng nhau
   - **(b)** Mọi điểm trùng nhau
   - **(c)** Khoảng cách tới láng giềng gần nhất tiến về 0
   - **(d)** Không có gì thay đổi

14. **[t14]** KNN suy giảm vì có nhiều feature vô ích. Cách khắc phục hợp lý? *(chọn tất cả đáp án đúng)*
   - **(a)** Feature selection
   - **(b)** Giảm chiều (PCA)
   - **(c)** Tăng số feature nhiễu
   - **(d)** Học metric / gán trọng số feature

15. **[t15]** KNN regressor (weights='uniform') có thể dự đoán giá trị lớn hơn mọi giá trị y trong tập train. *(Đúng/Sai)*

16. **[t16]** Tìm láng giềng trong 100 triệu embedding với độ trễ vài mili-giây. Lựa chọn hợp lý?
   - **(a)** KNN brute force
   - **(b)** Approximate nearest neighbor (HNSW, FAISS)
   - **(c)** KD-tree trên 768 chiều
   - **(d)** Tăng k

In [ ]:
theory_answers = {
    "t01": "b",  # a/b/c/d
    "t02": True,  # True/False
    "t03": "b",  # a/b/c/d
    "t04": "b",  # a/b/c/d
    "t05": "a",  # a/b/c/d
    "t06": "c",  # a/b/c/d
    "t07": False,  # True/False
    "t08": "a",  # a/b/c/d
    "t09": "b",  # a/b/c/d
    "t10": "ad",  # chọn nhiều, vd "ac"
    "t11": "b",  # a/b/c/d
    "t12": True,  # True/False
    "t13": "a",  # a/b/c/d
    "t14": "abd",  # chọn nhiều, vd "ac"
    "t15": False,  # True/False
    "t16": "b",  # a/b/c/d
}
cp.grade("theory", theory_answers)

### 💡 Giải thích — Theory Quiz

**[t01]** Đáp án: `b` — fit O(1), predict O(n·d).

**[t02]** Đáp án: `True` — Model chính là toàn bộ dữ liệu train.

**[t03]** Đáp án: `b` — k lớn → ranh giới mượt (bias cao, variance thấp).

**[t04]** Đáp án: `b` — Láng giềng gần nhất của mỗi điểm train là chính nó.

**[t05]** Đáp án: `a` — Văn bản dài gấp 3 vẫn cùng chủ đề → cosine distance = 0.

**[t06]** Đáp án: `c` — Tính khoảng cách tới n điểm, mỗi điểm O(d).

**[t07]** Đáp án: `False` — Khi d lớn (> ~20), KD-tree suy giảm về brute force (curse of dimensionality).

**[t08]** Đáp án: `a` — Giảm nhạy với k và giải quyết hoà phiếu.

**[t09]** Đáp án: `b` — Đổi đơn vị = đổi láng giềng.

**[t10]** Đáp án: `ad` — Tree và OLS (dự đoán) bất biến với phép nhân hằng số một feature; mọi model dựa trên khoảng cách đều thay đổi.

**[t11]** Đáp án: `b` — Accuracy train luôn chọn k = 1; chọn trên test là leakage.

**[t12]** Đáp án: `True` — Không thể chia đều số lẻ phiếu cho 2 lớp.

**[t13]** Đáp án: `a` — Tỉ lệ min/max → 1: 'láng giềng gần nhất' mất ý nghĩa.

**[t14]** Đáp án: `abd` — Mục tiêu: khoảng cách chỉ phản ánh feature có ích.

**[t15]** Đáp án: `False` — Dự đoán là trung bình của các y đã thấy → nằm trong [min y, max y].

**[t16]** Đáp án: `b` — Chấp nhận sai số nhỏ để nhanh hơn hàng nghìn lần.

## Phần 2 — Math Quiz

1. **[m01]** Khoảng cách Euclid giữa (1, 2) và (4, 6)? *(làm tròn 3 chữ số thập phân)*

2. **[m02]** Khoảng cách Manhattan giữa (1, 2) và (4, 6)? *(làm tròn 2 chữ số thập phân)*

3. **[m03]** Cosine distance giữa (1, 0) và (1, 1)? (1 − cos θ) *(làm tròn 3 chữ số thập phân)*

4. **[m04]** 3 láng giềng gần nhất có y = 10, 14, 30. Dự đoán của KNN regressor (uniform)? *(làm tròn 2 chữ số thập phân)*

5. **[m05]** 2 láng giềng: y = 10 ở khoảng cách 1, y = 20 ở khoảng cách 4. Dự đoán với trọng số 1/d? *(làm tròn 2 chữ số thập phân)*

6. **[m06]** Dự đoán 1 000 điểm với 10 000 điểm train: phải tính bao nhiêu khoảng cách (brute force)? *(làm tròn 0 chữ số thập phân)*

7. **[m07]** Quy tắc kinh nghiệm k ≈ √n với n = 900 mẫu cho k ≈ ? *(làm tròn 0 chữ số thập phân)*

In [ ]:
math_answers = {
    "m01": 5.0,  # số
    "m02": 7.0,  # số
    "m03": 0.2929,  # số
    "m04": 18.0,  # số
    "m05": 12.0,  # số
    "m06": 10000000.0,  # số
    "m07": 30.0,  # số
}
cp.grade("math", math_answers)

### 💡 Giải thích — Math Quiz

**[m01]** Đáp án: `5` — √(9 + 16) = 5.

**[m02]** Đáp án: `7` — 3 + 4 = 7.

**[m03]** Đáp án: `0.2929` — 1 − 1/√2 ≈ 0.293.

**[m04]** Đáp án: `18` — (10 + 14 + 30)/3 = 18.

**[m05]** Đáp án: `12` — (10·1 + 20·0.25)/(1 + 0.25) = 15/1.25 = 12.

**[m06]** Đáp án: `10000000` — 1 000 × 10 000 = 10⁷.

**[m07]** Đáp án: `30` — √900 = 30; bài 2 lớp nên chọn số lẻ gần đó (29 hoặc 31) rồi tinh chỉnh bằng CV.

## Phần 3 — Coding Exercise

### Task c01 — Ma trận khoảng cách vector hoá

Viết `distance_matrix(A, B)` trả về ma trận khoảng cách Euclid $(n_A \times n_B)$ **không dùng vòng lặp**,
dùng $\lVert a - b\rVert^2 = \lVert a\rVert^2 + \lVert b\rVert^2 - 2a^\top b$ và chặn giá trị âm trước khi lấy căn.

In [ ]:
def distance_matrix(A, B):
    ### START CODE HERE ### (≈ 2 dòng)
    sq = (A**2).sum(1)[:, None] + (B**2).sum(1)[None, :] - 2 * A @ B.T
    D = np.sqrt(np.maximum(sq, 0))
    ### END CODE HERE ###
    return D

In [ ]:
with cp.task("c01"):
    from scipy.spatial.distance import cdist
    rng = np.random.default_rng(1)
    A, B = rng.normal(size=(20, 4)), rng.normal(size=(15, 4))
    check_close(distance_matrix(A, B), cdist(A, B), tol=1e-8, message="khớp cdist")
    check_true(np.all(np.isfinite(distance_matrix(A, A))), "không NaN trên đường chéo")

### Task c02 — KNN classifier

Viết `knn_predict(X_train, y_train, X_test, k)` (nhãn nguyên 0..C−1, bỏ phiếu uniform; khi hoà chọn nhãn nhỏ hơn).
Có thể dùng `distance_matrix` ở trên.

In [ ]:
def knn_predict(X_train, y_train, X_test, k=5):
    ### START CODE HERE ### (≈ 4 dòng)
    D = distance_matrix(X_test, X_train)
    idx = np.argsort(D, axis=1, kind="stable")[:, :k]
    n_classes = int(y_train.max()) + 1
    pred = np.array([np.bincount(y_train[r], minlength=n_classes).argmax() for r in idx])
    ### END CODE HERE ###
    return pred

In [ ]:
with cp.task("c02"):
    from sklearn.neighbors import KNeighborsClassifier
    from ml_course.datasets import load_digits_data
    X, y, _ = load_digits_data()
    Xtr, ytr, Xte, yte = X[:1200], y[:1200], X[1200:], y[1200:]
    ours = knn_predict(Xtr, ytr, Xte, k=3)
    sk = KNeighborsClassifier(3).fit(Xtr, ytr).predict(Xte)
    check_greater(np.mean(ours == sk), 0.99, "dự đoán trùng sklearn ≥ 99%")
    check_equal(list(knn_predict(np.array([[0.0], [1.0], [10.0]]), np.array([0, 1, 1]), np.array([[0.2], [9.0]]), k=1)), [0, 1], "ví dụ nhỏ")

## Phần 4 — Model Analysis

Một nhóm dùng KNN (k = 1, không scaling) để dự đoán khách hàng có mua bảo hiểm hay không, với 120 feature
(tuổi, thu nhập theo đồng, 100 cột one-hot tỉnh thành, 18 cột hành vi). Họ báo cáo accuracy 100% trên tập train,
và chọn k = 1 vì "train accuracy cao nhất". Model chạy 4 giây cho mỗi dự đoán trên 3 triệu khách hàng.
Hãy chỉ ra các vấn đề và đề xuất cách sửa.

**Rubric tự chấm** (đạt khi ≥ 80% số mục):
- [ ] Chỉ ra accuracy train 100% với k = 1 là hiển nhiên (mỗi điểm là láng giềng của chính nó) → phải đánh giá bằng CV/test
- [ ] Chỉ ra thiếu scaling: thu nhập (đơn vị đồng) áp đảo khoảng cách → scale trong Pipeline
- [ ] Chỉ ra 120 feature (nhiều one-hot) → curse of dimensionality / feature vô ích → feature selection, giảm chiều, hoặc đổi model
- [ ] Chọn k (và weights, metric) bằng cross-validation trên tập train, không phải train accuracy
- [ ] Chỉ ra vấn đề độ trễ dự đoán O(n·d) với 3 triệu bản ghi → ANN (HNSW/FAISS), giảm mẫu, hoặc model khác (tree/linear)

✍️ **Bài mẫu:**

(1) Accuracy train 100% với k = 1 là tất yếu — vô nghĩa; phải dùng CV/test.
(2) Không scaling: thu nhập theo đồng có thang đo hàng triệu → khoảng cách gần như chỉ do thu nhập; cần StandardScaler trong Pipeline.
(3) 120 chiều với 100 cột one-hot tỉnh thành → khoảng cách loãng, curse of dimensionality; cần chọn feature/giảm chiều, mã hoá tỉnh bằng feature có ý nghĩa (vùng, thu nhập trung bình tỉnh...).
(4) Chọn k, weights, metric bằng GridSearchCV trên tập train.
(5) 4 s/dự đoán trên 3 triệu bản ghi: KNN brute force không phù hợp sản phẩm → ANN, condensed NN, hoặc chuyển sang Gradient Boosting/Logistic Regression.

In [ ]:
# Đổi False → True cho mục mà câu trả lời của bạn ĐÃ đáp ứng (trung thực!)
analysis_checklist = {
    "Chỉ ra accuracy train 100% với k = 1 là hiển nhiên (mỗi điểm là láng giềng của chính nó) → phải đánh giá bằng CV/test": True,
    "Chỉ ra thiếu scaling: thu nhập (đơn vị đồng) áp đảo khoảng cách → scale trong Pipeline": True,
    "Chỉ ra 120 feature (nhiều one-hot) → curse of dimensionality / feature vô ích → feature selection, giảm chiều, hoặc đổi model": True,
    "Chọn k (và weights, metric) bằng cross-validation trên tập train, không phải train accuracy": True,
    "Chỉ ra vấn đề độ trễ dự đoán O(n·d) với 3 triệu bản ghi → ANN (HNSW/FAISS), giảm mẫu, hoặc model khác (tree/linear)": True,
}
cp.self_assess("analysis", analysis_checklist)

## Phần 5 — Mini Project

**Mini project — KNN cho dữ liệu churn.** Đọc `customer_churn.csv` (`repo_path("data", "sample", "customer_churn.csv")`),
target `churn`. Chia train/test 75/25 (`random_state=0`, `stratify`). Xây Pipeline có tiền xử lý (scale cột số, one-hot cột phân loại),
KNN, chọn `n_neighbors`/`weights` bằng `GridSearchCV` (5-fold, `roc_auc`) trên tập train.
Đạt **ROC-AUC test ≥ 0.76** (biến `test_auc`).

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from ml_course.utils import repo_path

df = pd.read_csv(repo_path("data", "sample", "customer_churn.csv"))
### START CODE HERE ### (≈ 10 dòng)
num = ["tenure_months", "monthly_charges", "support_calls", "n_products"]
cat = ["contract", "internet_service", "payment_method"]
X_tr, X_te, y_tr, y_te = train_test_split(df[num + cat], df["churn"], test_size=0.25, random_state=0, stratify=df["churn"])
pre = ColumnTransformer([("num", StandardScaler(), num), ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
grid = GridSearchCV(make_pipeline(pre, KNeighborsClassifier()),
                    {"kneighborsclassifier__n_neighbors": [31, 61, 101, 151], "kneighborsclassifier__weights": ["uniform", "distance"]},
                    cv=5, scoring="roc_auc").fit(X_tr, y_tr)
test_auc = roc_auc_score(y_te, grid.predict_proba(X_te)[:, 1])
### END CODE HERE ###
print(grid.best_params_, "ROC-AUC test:", round(test_auc, 4))

In [ ]:
with cp.task("project"):
    check_greater(test_auc, 0.76, "ROC-AUC test ≥ 0.76")

## 🏁 Kết quả

In [ ]:
cp.report()